# AI Marketing Studio

**Mode:** Protected live OpenAI  
**Level:** Capstone  
**Estimated time:** 75 minutes

Building software is cheaper than it used to be; earning attention and trust is
not. This flagship capstone turns a finished product into a bounded campaign,
with real multimodal analysis, structured model responses, claims review,
persisted HITL, and an early-signal learning pass.


## Problem and outcome

Relayboard exists, but its team has not found positioning or distribution. The
studio must identify a defensible audience, choose a message, create three
channel assets, reject unsupported claims, obtain a human-approved claims tool
execution, and use early campaign signals to select the next experiment.

The output is a launch kit, not loose copy: audience hypotheses, positioning,
message hierarchy, landing-page hero, email, social copy, channel plan,
approved claims, measurement plan, and next experiment.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "examples" / "notebooks"]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, setup_case_study, show_artifact, show_callout,
    show_flow, show_json, show_message_graph, show_roles, show_scorecard,
    show_spore, show_timeline, stage,
)

setup_case_study("AI Marketing Studio")


## Course prerequisites

Complete the Reef pipeline, feedback loop, tools, memory, ModelRuntime, HITL,
and multimodal lessons. This notebook requires `OPENAI_API_KEY` and
`PRAVAL_OPENAI_MODEL`. It makes eight bounded provider calls and is certified
only by the protected manual live workflow.


## Architecture and responsibilities

```text
campaign_brief → research_insight × 2 → positioning_proposal → channel_plan
               → asset_request × 3 → asset_draft → claims_review
               → creative_review → revised_campaign → HITL approval/resume
               → approved_campaign → signal_analysis → optimized_campaign
```

Every model result is a schema-validated object before it becomes a Spore.


In [ ]:
show_flow(
    ("Brief + screenshot", "bounded product truth", "human"),
    ("Research", "two independent lenses", "agent"),
    ("Strategy", "positioning and channels", "agent"),
    ("Studio", "three launch assets", "provider"),
    ("Claims + creative", "review and revise", "tool"),
    ("Human approval", "persist, edit, resume", "human"),
    ("Learning", "next experiment", "memory"),
)
show_roles([
    ("Campaign director", "Own brief, correlation, memory, and launch kit", "agent"),
    ("Audience researcher", "Analyze product and screenshot", "agent"),
    ("Customer-insight strategist", "Interpret interview language", "agent"),
    ("Positioning strategist", "Create a defensible market position", "agent"),
    ("Channel planner", "Turn strategy into bounded distribution", "agent"),
    ("Content studio", "Draft landing, email, and social assets", "agent"),
    ("Claims reviewer", "Check facts and request protected approval", "agent"),
    ("Creative director", "Perform one bounded creative revision", "agent"),
    ("Human approval boundary", "Edit and approve protected tool arguments", "human"),
    ("Learning analyst", "Read early signals and select next experiment", "agent"),
])


## Design decisions

- Product facts, interviews, brand rules, channel limits, screenshot, and signals
  are committed fixtures.
- Six ordinary structured calls plus HITL interruption and resume cap the live
  run at eight provider calls.
- The claims tool is approval-protected and receives edited arguments.
- Early campaign signals are bounded evidence; the learning tool owns arithmetic
  while the final launch kit retains strategic context.


## Implementation

Load and display the actual product screenshot before constructing Agents. The
first research request sends those bytes through the provider-neutral
`ContentPart` interface.


In [ ]:
import base64
import json
import sqlite3
import subprocess
import sys
import tempfile

from IPython.display import Image, display
from praval import Agent, ContentPart, InterventionRequired, ToolSpec, get_reef
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

values = require_env("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL")
reset_reef(); reset_tool_registry()
fixture_root = find_example_asset("notebooks/fixtures/marketing")


def load_json(name):
    return json.loads((fixture_root / name).read_text(encoding="utf-8"))


product_brief = (fixture_root / "product_brief.md").read_text()
brand_guide = (fixture_root / "brand_guide.md").read_text()
interviews = load_json("customer_interviews.json")
facts = load_json("approved_facts.json")
constraints = load_json("channel_constraints.json")
signals = load_json("campaign_signals.json")
image_bytes = base64.b64decode((fixture_root / "product_screenshot.png.base64").read_text())
workspace = tempfile.TemporaryDirectory(prefix="praval-marketing-capstone-")
image_path = Path(workspace.name) / "relayboard.png"
image_path.write_bytes(image_bytes)
display(Image(filename=str(image_path), width=720))


In [ ]:
research_schema = {"type": "object", "properties": {
    "insights": {"type": "array", "items": {"type": "string"}},
    "risks": {"type": "array", "items": {"type": "string"}},
    "evidence": {"type": "array", "items": {"type": "string"}}},
    "required": ["insights", "risks", "evidence"], "additionalProperties": False}
strategy_schema = {"type": "object", "properties": {
    "audience": {"type": "string"}, "problem": {"type": "string"},
    "promise": {"type": "string"},
    "proof_points": {"type": "array", "items": {"type": "string"}},
    "counter_positioning": {"type": "string"},
    "channels": {"type": "array", "items": {"type": "string"}},
    "channel_rationale": {"type": "string"},
    "budget_split": {"type": "array", "items": {"type": "string"}},
    "measurement_plan": {"type": "array", "items": {"type": "string"}}},
    "required": ["audience", "problem", "promise", "proof_points", "counter_positioning",
                 "channels", "channel_rationale", "budget_split", "measurement_plan"],
    "additionalProperties": False}
asset_properties = {name: {"type": "string"} for name in
                    ("hero", "subhead", "email_subject", "email_body", "linkedin_post", "paid_social")}
asset_schema = {"type": "object", "properties": asset_properties,
                "required": list(asset_properties), "additionalProperties": False}
review_schema = {"type": "object", "properties": {
    "decision": {"type": "string", "enum": ["approved", "revise"]},
    "approved_claims": {"type": "array", "items": {"type": "string"}},
    "rejected_claims": {"type": "array", "items": {"type": "string"}},
    "rationale": {"type": "string"}},
    "required": ["decision", "approved_claims", "rejected_claims", "rationale"],
    "additionalProperties": False}
creative_schema = {"type": "object", "properties": {
    **asset_properties, "revision_reason": {"type": "string"},
    "winning_message": {"type": "string"}, "failed_hypothesis": {"type": "string"},
    "next_experiment": {"type": "string"}},
    "required": [*asset_properties, "revision_reason", "winning_message",
                 "failed_hypothesis", "next_experiment"], "additionalProperties": False}


In [ ]:
config = {"temperature": 0, "max_output_tokens": 500, "timeout": 90, "retries": 2}


def make_agent(name, system, *, memory=False, hitl=False, db_path=None):
    return Agent(
        name, provider="openai", model=values["PRAVAL_OPENAI_MODEL"],
        system_message=system, config=config, memory_enabled=memory,
        memory_config={"backend": "memory"} if memory else None,
        hitl_enabled=hitl, hitl_db_path=db_path,
    )


hitl_db = str(Path(workspace.name) / "marketing-hitl.sqlite3")
director = make_agent("campaign-director", "Own the campaign contract.", memory=True)
audience_agent = make_agent("audience-researcher", "Extract defensible audience insights.")
customer_agent = make_agent("customer-insight-strategist", "Use only interview evidence.")
positioning_agent = make_agent("positioning-strategist", "Create precise positioning and channel strategy.")
channel_agent = make_agent("channel-planner", "Own channel constraints and distribution logic.")
content_agent = make_agent("content-studio", "Write restrained, factual launch assets.")
claims_agent = make_agent("claims-reviewer", "Reject claims outside approved facts.", hitl=True, db_path=hitl_db)
creative_agent = make_agent("creative-director", "Revise once for clarity and brand fit.")
learning_agent = make_agent("learning-analyst", "Turn bounded campaign signals into an experiment.")
all_agents = [director, audience_agent, customer_agent, positioning_agent, channel_agent,
              content_agent, claims_agent, creative_agent, learning_agent]


In [ ]:
REQUIRED = {"type", "domain_id", "correlation_id", "producer", "stage", "status", "payload"}
marketing_messages, received_spores, provider_evidence, claim_executions = [], [], [], []
channel = "ai-marketing-studio"
reef = get_reef()
reef.create_channel(channel)


def marketing_message(message_type, producer, stage_name, status, payload):
    message = {
        "type": message_type, "domain_id": "relayboard-launch",
        "correlation_id": "campaign-relayboard-001", "producer": producer,
        "stage": stage_name, "status": status, "payload": payload,
    }
    assert set(message) == REQUIRED
    return message


def capture(spore):
    received_spores.append(spore)
    marketing_messages.append(dict(spore.knowledge))


reef.subscribe("campaign-audit", capture, channel=channel)


def emit(sender, message_type, stage_name, status, payload):
    sender.broadcast_knowledge(
        marketing_message(message_type, sender.name, stage_name, status, payload),
        channel=channel,
    )
    assert reef.wait_for_completion(timeout=30)


def record(response, stage_name):
    assert response.provider == "openai" and response.model
    assert response.usage is not None and response.usage.total_tokens > 0
    provider_evidence.append({"stage": stage_name, "provider": response.provider,
                              "model": response.model, "usage": response.usage.model_dump()})
    return json.loads(response.content)


emit(director, "campaign_brief", "brief", "open",
     {"product": "Relayboard", "goal": "40 qualified design partners in four weeks"})


In [ ]:
audience_response = await audience_agent.agenerate(
    [
        ContentPart.text_part(
            "Analyze this product brief and actual product screenshot. Return audience insights, "
            "risks, and evidence. Do not invent metrics.\n" + product_brief
        ),
        ContentPart.image_base64(base64.b64encode(image_bytes).decode("ascii"), "image/png"),
    ],
    response_schema=research_schema,
    metadata={"capstone": "marketing", "stage": "multimodal-audience"},
)
audience = record(audience_response, "multimodal-audience")
assert audience["insights"] and audience["evidence"]
emit(audience_agent, "research_insight", "audience_research", "complete",
     {**audience, "multimodal": True})

customer_response = await customer_agent.agenerate(
    "Extract customer-language insights from this bounded interview JSON. "
    "Return evidence strings that identify the interview role.\n" + json.dumps(interviews),
    response_schema=research_schema,
    metadata={"capstone": "marketing", "stage": "customer-insight"},
)
customer = record(customer_response, "customer-insight")
emit(customer_agent, "research_insight", "customer_research", "complete", customer)


In [ ]:
strategy_prompt = """Create positioning and a channel plan from the bounded materials below.
Respect every brand and channel constraint. Use only approved facts.

AUDIENCE RESEARCH:
{audience}

CUSTOMER RESEARCH:
{customer}

APPROVED FACTS:
{facts}

BRAND:
{brand}

CHANNEL CONSTRAINTS:
{constraints}
""".format(audience=json.dumps(audience), customer=json.dumps(customer),
           facts=json.dumps(facts), brand=brand_guide, constraints=json.dumps(constraints))
strategy_response = await positioning_agent.agenerate(
    strategy_prompt, response_schema=strategy_schema,
    metadata={"capstone": "marketing", "stage": "positioning-and-channels"},
)
strategy = record(strategy_response, "positioning-and-channels")
emit(positioning_agent, "positioning_proposal", "positioning", "complete",
     {key: strategy[key] for key in ("audience", "problem", "promise", "proof_points", "counter_positioning")})
emit(channel_agent, "channel_plan", "distribution", "complete",
     {key: strategy[key] for key in ("channels", "channel_rationale", "budget_split", "measurement_plan")})

for asset_type in ("landing_page", "email", "social"):
    emit(director, "asset_request", "content_request", "open", {"asset_type": asset_type})


In [ ]:
asset_response = await content_agent.agenerate(
    "Create one landing hero, email, LinkedIn post, and paid-social line. "
    "Follow the exact facts, brand, constraints, and strategy.\n" +
    json.dumps({"facts": facts, "constraints": constraints, "strategy": strategy}) +
    "\nBRAND:\n" + brand_guide,
    response_schema=asset_schema,
    metadata={"capstone": "marketing", "stage": "asset-draft"},
)
assets = record(asset_response, "asset-draft")
assert len(assets["hero"]) <= constraints["landing_page"]["headline_max_chars"]
assert len(assets["email_subject"]) <= constraints["email"]["subject_max_chars"]
emit(content_agent, "asset_draft", "content", "needs_review", assets)

claims_response = await claims_agent.agenerate(
    "Review these assets against approved and prohibited claims. Return a strict decision.\n" +
    json.dumps({"assets": assets, "facts": facts}),
    response_schema=review_schema,
    metadata={"capstone": "marketing", "stage": "claims-review"},
)
claims_review = record(claims_response, "claims-review")
emit(claims_agent, "claims_review", "claims", claims_review["decision"], claims_review)


In [ ]:
creative_response = await creative_agent.agenerate(
    "Perform exactly one creative revision. Preserve approved facts, apply the brand guide, "
    "and make the decision-confidence message primary. Return revised assets plus a winning "
    "message, failed hypothesis, and next experiment.\n" +
    json.dumps({"draft": assets, "claims_review": claims_review, "signals": signals,
                "strategy": strategy}) + "\nBRAND:\n" + brand_guide,
    response_schema=creative_schema,
    metadata={"capstone": "marketing", "stage": "creative-revision"},
)
revised = record(creative_response, "creative-revision")
emit(creative_agent, "creative_review", "creative_review", "revise",
     {"reason": revised["revision_reason"], "revision_count": 1})
emit(creative_agent, "revised_campaign", "revision", "complete", revised)


@tool("marketing_signal_analysis", description="Compute bounded campaign conversion signals.",
      category="marketing", shared=True)
def analyze_signals(variants: list) -> dict:
    scored = [{**item, "qualified_rate": item["qualified_signups"] / item["impressions"]}
              for item in variants]
    winner = max(scored, key=lambda item: item["qualified_rate"])
    return {"winner": winner["id"], "qualified_rate": winner["qualified_rate"],
            "comparison": scored}


signal_analysis = analyze_signals.execute_as_tool(signals["variants"])


In [ ]:
approval_spec = ToolSpec(
    name="approve_campaign_claims",
    description="Approve the exact final claims before campaign activation. Always call this tool.",
    parameters={"type": "object", "properties": {
        "claims": {"type": "array", "items": {"type": "string"}}},
        "required": ["claims"], "additionalProperties": False},
    strict=True, requires_approval=True, risk_level="high",
    approval_reason="A human must approve externally published product claims.",
)


def approve_campaign_claims(claims: list) -> str:
    invalid = [claim for claim in claims if claim not in facts["approved_claims"]]
    if invalid:
        raise ValueError("claim is outside the approved fact set")
    claim_executions.append(list(claims))
    return json.dumps({"approved": True, "claims": claims})


claims_agent.add_tool_spec(approval_spec, approve_campaign_claims)
approval_schema = {"type": "object", "properties": {
    "status": {"type": "string", "enum": ["approved"]},
    "claims": {"type": "array", "items": {"type": "string"}}},
    "required": ["status", "claims"], "additionalProperties": False}

try:
    await claims_agent.agenerate(
        "You must call approve_campaign_claims with the final campaign claims. "
        "Do not approve them yourself. After the tool succeeds, return the approval object.\n" +
        json.dumps({"revised_campaign": revised, "candidate_claims": facts["approved_claims"]}),
        response_schema=approval_schema,
        metadata={"capstone": "marketing", "stage": "protected-final-approval"},
    )
except InterventionRequired as interruption:
    pending = claims_agent.get_pending_interventions(run_id=interruption.run_id)
    assert len(pending) == 1 and pending[0].tool_name == "approve_campaign_claims"
else:
    raise AssertionError("The real model did not produce the required protected tool call")


In [ ]:
inspection_code = (
    'import json, sqlite3, sys; c=sqlite3.connect(sys.argv[1]); '
    'p=c.execute("select count(*) from interventions where status=\'PENDING\'").fetchone()[0]; '
    'r=c.execute("select count(*) from suspended_runs where status=\'pending\'").fetchone()[0]; '
    'print(json.dumps({"pending_interventions": p, "suspended_runs": r}))'
)
persisted = subprocess.run(
    [sys.executable, "-c", inspection_code, hitl_db], shell=False,
    check=True, capture_output=True, text=True, timeout=15,
)
persisted_state = json.loads(persisted.stdout)
assert persisted_state == {"pending_interventions": 1, "suspended_runs": 1}
show_json({
    "tool": pending[0].tool_name, "original_arguments": pending[0].original_args,
    "persisted_state": persisted_state, "database": "temporary SQLite",
}, "Pending campaign approval")

edited_claims = facts["approved_claims"][:2]
claims_agent.approve_intervention(
    pending[0].id, reviewer="campaign-owner", edited_args={"claims": edited_claims}
)
approval_text = await claims_agent.aresume_run(pending[0].run_id)
approval = json.loads(approval_text)
assert approval["status"] == "approved" and claim_executions == [edited_claims]
emit(claims_agent, "approved_campaign", "human_approval", "approved",
     {"claims": edited_claims, "intervention_id": pending[0].id})


## End-to-end run

The six ordinary structured calls, interrupted approval call, and resumed call
form the eight-call budget. The learning pass now uses a registered arithmetic
tool, persists what was learned, and assembles the final launch contract.


In [ ]:
emit(learning_agent, "signal_analysis", "learning", "complete", signal_analysis)
for content in (
    "Winning message: " + revised["winning_message"],
    "Failed hypothesis: " + revised["failed_hypothesis"],
    "Next experiment: " + revised["next_experiment"],
):
    director.remember(content, 0.9)

launch_kit = {
    "audience_hypotheses": audience["insights"] + customer["insights"],
    "positioning": {key: strategy[key] for key in
                    ("audience", "problem", "promise", "proof_points", "counter_positioning")},
    "message_hierarchy": [revised["winning_message"], strategy["promise"]],
    "landing_page": {"hero": revised["hero"], "subhead": revised["subhead"]},
    "email": {"subject": revised["email_subject"], "body": revised["email_body"]},
    "social": {"linkedin": revised["linkedin_post"], "paid": revised["paid_social"]},
    "channel_plan": {key: strategy[key] for key in
                     ("channels", "channel_rationale", "budget_split")},
    "approved_claims": edited_claims,
    "measurement_plan": strategy["measurement_plan"],
    "signal_analysis": signal_analysis,
    "next_experiment": revised["next_experiment"],
}
emit(director, "optimized_campaign", "optimization", "complete", launch_kit)
assert reef.wait_for_completion(timeout=30)


## Inspect the system

Certification checks behavior, not literary quality: real provider metadata,
nonzero usage, actual image input, a model-generated protected tool call,
persisted interruption and resume, schema-valid assets, memory, and a complete
artifact contract.


In [ ]:
memory = [entry.content for entry in director.recall("winning failed next", limit=20)]
required_artifact = {
    "audience_hypotheses", "positioning", "message_hierarchy", "landing_page",
    "email", "social", "channel_plan", "approved_claims", "measurement_plan",
    "signal_analysis", "next_experiment",
}
assert set(launch_kit) == required_artifact
assert len(provider_evidence) == 6
assert all(item["provider"] == "openai" and item["usage"]["total_tokens"] > 0
           for item in provider_evidence)
assert any(item["type"] == "research_insight" and item["payload"].get("multimodal")
           for item in marketing_messages)
assert len(claim_executions) == 1 and len(memory) >= 3
assert {item["correlation_id"] for item in marketing_messages} == {"campaign-relayboard-001"}

show_message_graph(marketing_messages)
show_spore(next(spore for spore in received_spores if spore.knowledge["type"] == "approved_campaign"),
           "Resumed human-approval Spore")
show_json({"provider_calls_with_responses": provider_evidence,
           "persisted_hitl": persisted_state, "campaign_memory": memory},
          "Live certification evidence")
show_timeline()

show_scorecard([
    ("Structured provider responses", len(provider_evidence), "pass"),
    ("Multimodal screenshot", "executed", "pass"),
    ("Protected tool executions", len(claim_executions), "pass"),
    ("Persisted HITL", persisted_state["pending_interventions"], "pass"),
    ("Creative revisions", 1, "pass"),
    ("Campaign memories", len(memory), "pass"),
    ("Artifact fields", len(launch_kit), "pass"),
])
show_artifact("Relayboard launch kit", launch_kit)


## Failure modes and tradeoffs

Provider calls can fail because of authentication, rate limits, or capability
changes; the protected workflow retries only transient failures. Structural
assertions intentionally avoid judging copy by exact wording. The signal sample
is synthetic and bounded, so it demonstrates learning architecture rather than
statistical significance.

## Extensions

Run a second approved creative variant, connect real campaign analytics through
MCP, or add jurisdiction-specific compliance review. Preserve the fact boundary
and require a fresh intervention when approved claims change.

## Cleanup

All OpenAI Agents, Reef workers, memory, the HITL database, decoded screenshot,
and registered tools are released after the final inspection.


In [ ]:
director.memory.clear_agent_memories(director.name)
for live_agent in all_agents:
    live_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
workspace_root = Path(workspace.name)
workspace.cleanup()
assert not workspace_root.exists()
show_callout("Marketing Studio closed", "Provider, memory, Reef, HITL, and media resources were released.", role="reef")
